In [10]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import requests
from pathlib import Path
from src.config import UCDP_DIR, CHECKPOINT_DIR, SIPRI_DIR
from src.io_utils import save_checkpoint, checkpoint_exists

In [11]:
dest = UCDP_DIR / "ucdp_acd.csv"
print("File exists:", dest.exists())
print("File size:", dest.stat().st_size, "bytes")

File exists: True
File size: 506186 bytes


In [12]:
acd = pd.read_csv(UCDP_DIR / "ucdp_acd.csv")
print(acd.shape)
print(acd.columns.tolist())
acd.head(3)

(2752, 28)
['conflict_id', 'location', 'side_a', 'side_a_id', 'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility', 'territory_name', 'year', 'intensity_level', 'cumulative_intensity', 'type_of_conflict', 'start_date', 'start_prec', 'start_date2', 'start_prec2', 'ep_end', 'ep_end_date', 'ep_end_prec', 'gwno_a', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'region', 'version']


,conflict_id,location,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,territory_name,...,ep_end,ep_end_date,ep_end_prec,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,gwno_loc,region,version
0,11342,India,Government of India,141,NaN,GNLA,1163,NaN,1,Garoland,...,1,2012-12-21,NaN,750,NaN,NaN,NaN,750,3,25.1
1,11342,India,Government of India,141,NaN,GNLA,1163,NaN,1,Garoland,...,1,2014-11-27,NaN,750,NaN,NaN,NaN,750,3,25.1
2,11343,"Egypt, Israel",Government of Egypt,117,NaN,Government of Israel,121,NaN,1,Suez/Sinai,...,1,1967-06-10,NaN,651,NaN,666,NaN,"651, 666",2,25.1


In [13]:
save_checkpoint(acd, CHECKPOINT_DIR / "ucdp_acd_raw.parquet")

[checkpoint] saved → ucdp_acd_raw.parquet  (2,752 rows)


In [14]:
print("Year range:", acd['year'].min(), "→", acd['year'].max())
print("\nConflict types:")
print(acd['type_of_conflict'].value_counts().sort_index())
print("\nIntensity levels:")
print(acd['intensity_level'].value_counts().sort_index())

Year range: 1946 → 2024

Conflict types:
type_of_conflict
1     117
2     147
3    2004
4     484
Name: count, dtype: int64

Intensity levels:
intensity_level
1    2066
2     686
Name: count, dtype: int64


In [15]:
import openpyxl
wb = openpyxl.load_workbook(SIPRI_DIR / "sipri_milex.xlsx", read_only=True)
print(wb.sheetnames)

['Front page', 'Regional totals', 'Local currency financial years', 'Local currency calendar years', 'Constant (2024) US$', 'Current US$', 'Share of GDP', 'Per capita', 'Share of Govt. spending', 'Footnotes']


In [16]:
milex_raw = pd.read_excel(
    SIPRI_DIR / "sipri_milex.xlsx",
    sheet_name="Constant (2024) US$",
    skiprows=5        # SIPRI has header rows before the actual data
)
print(milex_raw.shape)
print(milex_raw.columns.tolist()[:10])  # first 10 columns
milex_raw.head(3)

(193, 80)
['Country', 'Unnamed: 1', 'Notes', 1949, 1950, 1951, 1952, 1953, 1954, 1955]


,Country,Unnamed: 1,Notes,1949,1950,1951,1952,1953,1954,1955,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,North Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Drop junk columns
milex = milex_raw.drop(columns=['Unnamed: 1', 'Notes'])

# Year columns are integers
year_cols = [c for c in milex.columns if isinstance(c, int)]

# Drop region header rows — they have no year data at all
milex = milex.dropna(subset=year_cols, how='all')

# Drop any remaining rows where Country is NaN
milex = milex.dropna(subset=['Country'])

# Reset index
milex = milex.reset_index(drop=True)

print(milex.shape)
print(milex['Country'].head(20).tolist())

(174, 78)
['Algeria', 'Libya', 'Morocco', 'Tunisia', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 'Cape Verde', 'Central African Republic', 'Chad', 'Congo, DR', 'Congo, Republic', "Cote d'Ivoire", 'Djibouti', 'Equatorial Guinea', 'Eritrea', 'Ethiopia']


In [18]:
milex_long = milex.melt(
    id_vars=['Country'],
    value_vars=year_cols,
    var_name='year',
    value_name='milex_const2024_usd'
)

# Drop missing values (country didn't report that year)
milex_long = milex_long.dropna(subset=['milex_const2024_usd'])

# Fix types
milex_long['year'] = milex_long['year'].astype(int)

# Sort
milex_long = milex_long.sort_values(['Country', 'year']).reset_index(drop=True)

print(milex_long.shape)
print(milex_long.dtypes)
milex_long.head(5)

(13376, 3)
Country                   str
year                    int64
milex_const2024_usd    object
dtype: object


,Country,year,milex_const2024_usd
0,Afghanistan,1949,...
1,Afghanistan,1950,...
2,Afghanistan,1951,...
3,Afghanistan,1952,...
4,Afghanistan,1953,...


In [20]:
# Replace SIPRI placeholder strings with NaN, then convert to float
milex_long['milex_const2024_usd'] = pd.to_numeric(
    milex_long['milex_const2024_usd'], errors='coerce'
)

# Drop rows that are still NaN after coercion
milex_long = milex_long.dropna(subset=['milex_const2024_usd'])

print(milex_long.shape)
print(milex_long.dtypes)
print(milex_long['milex_const2024_usd'].describe())

save_checkpoint(milex_long, CHECKPOINT_DIR / "sipri_milex_long_raw.parquet")

(8435, 3)
Country                    str
year                     int64
milex_const2024_usd    float64
dtype: object
count    8.435000e+03
mean     1.232535e+04
std      6.986929e+04
min      0.000000e+00
25%      1.297652e+02
50%      9.045244e+02
75%      4.708165e+03
max      1.061674e+06
Name: milex_const2024_usd, dtype: float64
[checkpoint] saved → sipri_milex_long_raw.parquet  (8,435 rows)
